# Entrenamiento local: defectos en superficies metálicas

Clasificación multi-clase (5 etiquetas) sobre el dataset sintético local.

**Modelos:** ResNet-18 · EfficientNet-B0 · MobileNetV3-Small  
**Datos:** `industrial_defect_dataset/train` y `val` (PNG 256×256, grayscale → RGB para ImageNet)  
**Métricas:** accuracy, F1 macro, recall por clase, matriz de confusión

## 1. Configuración e imports

In [1]:
from __future__ import annotations

import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from torchvision.models import (
    EfficientNet_B0_Weights,
    MobileNet_V3_Small_Weights,
    ResNet18_Weights,
)
from tqdm.auto import tqdm

# --- Rutas ---
ROOT = Path(".").resolve()
DATA_DIR = ROOT / "industrial_defect_dataset"
OUTPUT_DIR = ROOT / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FIGURES_DIR = ROOT / "docs" / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# --- Hiperparámetros ---
SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 32
NUM_EPOCHS = 20
LR = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5  # early stopping sobre F1 macro (val)
NUM_WORKERS = 4  # paraleliza carga de datos para saturar GPU
MODELS_TO_TRAIN = ["resnet18", "efficientnet_b0", "mobilenet_v3_small"]

# --- GPU CUDA obligatoria ---
assert torch.cuda.is_available(), (
    'CUDA no disponible en este kernel. En Cursor/Jupyter selecciona el kernel '
    'Python 3.12 con PyTorch cu128 (torch.cuda.is_available() == True). '
    'Verifica en terminal: python -c "import torch; print(torch.__version__, torch.cuda.is_available())"'
)
DEVICE = torch.device("cuda:0")
torch.cuda.set_device(DEVICE)
_probe = torch.zeros(1, device=DEVICE)
assert _probe.is_cuda, "El tensor de prueba no está en CUDA"
del _probe

_props = torch.cuda.get_device_properties(0)
_total_gb = _props.total_memory / (1024**3)
_free_gb, _ = torch.cuda.mem_get_info(0)
print(f"Dispositivo: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA (runtime): {torch.version.cuda} | PyTorch: {torch.__version__}")
print(f"Capability: {_props.major}.{_props.minor}")
print(f"VRAM total: {_total_gb:.2f} GB | libre: {_free_gb / (1024**3):.2f} GB")


Dispositivo: cuda:0
GPU: NVIDIA GeForce RTX 5060 Ti
CUDA (runtime): 12.8 | PyTorch: 2.11.0+cu128
Capability: 12.0
VRAM total: 15.93 GB | libre: 14.78 GB


In [2]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(f"cuDNN benchmark=ON | entrenamiento en {DEVICE} ({torch.cuda.get_device_name(0)})")


def show_figure() -> None:
    """Muestra la figura si el backend es interactivo; si no, la cierra sin warning."""
    import matplotlib

    backend = matplotlib.get_backend().lower()
    interactive = matplotlib.is_interactive() and "agg" not in backend
    if interactive:
        plt.show()
    else:
        plt.close("all")


cuDNN benchmark=ON | entrenamiento en cuda:0 (NVIDIA GeForce RTX 5060 Ti)


## 2. Dataset y DataLoaders

Las imágenes son grayscale (`L`). Se convierten a RGB (3 canales) para usar pesos ImageNet.

In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.Grayscale(num_output_channels=3),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

train_ds = datasets.ImageFolder(DATA_DIR / "train", transform=train_transform)
val_ds = datasets.ImageFolder(DATA_DIR / "val", transform=val_transform)

assert train_ds.classes == val_ds.classes, "Las clases train/val deben coincidir"
CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
print("Clases:", CLASS_NAMES)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    **loader_kwargs,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

# Verificación: batch y modelo en GPU
_images, _labels = next(iter(train_loader))
_inputs = _images.to(DEVICE, non_blocking=True)
assert _inputs.is_cuda, "El batch no se movió a CUDA"
_probe_model = models.resnet18(weights=None)
_probe_model.fc = nn.Linear(_probe_model.fc.in_features, NUM_CLASSES)
_probe_model = _probe_model.to(DEVICE)
_param_device = next(_probe_model.parameters()).device
assert _param_device.type == "cuda", f"Modelo en {_param_device}, se esperaba CUDA"
print(f"Batch en GPU: {_inputs.device} shape={tuple(_inputs.shape)}")
print(f"Modelo en GPU: {_param_device}")
del _images, _labels, _inputs, _probe_model
torch.cuda.empty_cache()


Clases: ['crack', 'hole', 'normal', 'rust', 'scratch']
Train: 12000 | Val: 3000


Batch en GPU: cuda:0 shape=(32, 3, 256, 256)
Modelo en GPU: cuda:0


In [4]:
# Vista rápida de un batch (imágenes procesadas)
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, img, label in zip(axes.ravel(), images[:8], labels[:8]):
    x = img.permute(1, 2, 0).numpy()
    x = x * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    ax.imshow(np.clip(x, 0, 1))
    ax.set_title(CLASS_NAMES[int(label)])
    ax.axis("off")
plt.suptitle("Muestras de entrenamiento (batch procesado)")
plt.tight_layout()
sample_path = FIGURES_DIR / "sample_batch.png"
plt.savefig(sample_path, dpi=150, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "sample_batch.png", dpi=150, bbox_inches="tight")
show_figure()
print(f"Muestras guardadas en: {sample_path}")


Muestras guardadas en: G:\Maestria\Modulo 9\Proyecto Final\docs\figures\sample_batch.png


## 3. Modelos (transfer learning ImageNet)

In [5]:
def build_model(name: str, num_classes: int = NUM_CLASSES) -> nn.Module:
    name = name.lower()
    if name == "resnet18":
        model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
    elif name == "mobilenet_v3_small":
        model = models.mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, num_classes)
    else:
        raise ValueError(f"Modelo no soportado: {name}")
    return model


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


for name in MODELS_TO_TRAIN:
    m = build_model(name)
    print(f"{name:22s} params={count_parameters(m):,}")

resnet18               params=11,179,077
efficientnet_b0        params=4,013,953


mobilenet_v3_small     params=1,522,981


## 4. Funciones de entrenamiento y evaluación

In [6]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    for inputs, targets in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        total_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(targets.cpu().numpy())

    y_true = np.concatenate(all_labels)
    y_pred = np.concatenate(all_preds)
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: torch.amp.GradScaler,
) -> tuple[float, float]:
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, targets in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
    return running_loss / len(loader.dataset), correct / max(total, 1)


def train_model(model_name: str) -> dict:
    set_seed()
    model = build_model(model_name).to(DEVICE)
    param_device = next(model.parameters()).device
    assert param_device.type == "cuda", f"{model_name} no está en CUDA ({param_device})"
    print(f"Modelo {model_name} en: {param_device}")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = torch.amp.GradScaler("cuda")

    history = {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "val_f1": [],
    }
    best_f1 = -1.0
    best_state = None
    best_metrics = None
    epochs_no_improve = 0
    ckpt_path = CHECKPOINT_DIR / f"{model_name}_best.pt"

    print(f"\n=== Entrenando {model_name} en {DEVICE} (AMP) ===")
    t0 = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler
        )
        val_metrics = evaluate(model, val_loader)
        scheduler.step()

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])
        history["val_f1"].append(val_metrics["f1_macro"])

        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_f1={val_metrics['f1_macro']:.4f}"
        )

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_metrics = val_metrics
            epochs_no_improve = 0
            torch.save(
                {
                    "model_name": model_name,
                    "state_dict": best_state,
                    "class_names": CLASS_NAMES,
                    "metrics": {
                        "accuracy": best_metrics["accuracy"],
                        "f1_macro": best_metrics["f1_macro"],
                        "f1_weighted": best_metrics["f1_weighted"],
                        "val_loss": best_metrics["loss"],
                    },
                },
                ckpt_path,
            )
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping en epoch {epoch} (paciencia={PATIENCE})")
                break

    elapsed = time.time() - t0
    model.load_state_dict(best_state)
    final = evaluate(model, val_loader)

    hist_path = FIGURES_DIR / f"history_{model_name}.json"
    with open(hist_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    result = {
        "model": model_name,
        "params": count_parameters(model),
        "epochs_run": len(history["epoch"]),
        "seconds": elapsed,
        "accuracy": final["accuracy"],
        "f1_macro": final["f1_macro"],
        "f1_weighted": final["f1_weighted"],
        "val_loss": final["loss"],
        "checkpoint": str(ckpt_path),
        "history": history,
        "y_true": final["y_true"],
        "y_pred": final["y_pred"],
        "device": str(DEVICE),
    }
    print(
        f"Mejor {model_name}: acc={result['accuracy']:.4f} "
        f"f1_macro={result['f1_macro']:.4f} ({elapsed/60:.1f} min) | device={DEVICE}"
    )
    print(f"Historial guardado en: {hist_path}")
    return result


## 5. Entrenar los tres modelos

> En GPU suele tomar varios minutos por modelo. Ajusta `NUM_EPOCHS` / `BATCH_SIZE` si necesitas una prueba rápida.

In [7]:
results = []
for model_name in MODELS_TO_TRAIN:
    results.append(train_model(model_name))

summary = pd.DataFrame(
    [
        {
            "model": r["model"],
            "params": r["params"],
            "epochs_run": r["epochs_run"],
            "minutes": round(r["seconds"] / 60, 2),
            "accuracy": r["accuracy"],
            "f1_macro": r["f1_macro"],
            "f1_weighted": r["f1_weighted"],
            "val_loss": r["val_loss"],
            "checkpoint": r["checkpoint"],
        }
        for r in results
    ]
).sort_values("f1_macro", ascending=False)

summary_path = OUTPUT_DIR / "comparison_summary.csv"
summary.to_csv(summary_path, index=False)
print(summary.to_string(index=False))
print(f"\nResumen guardado en: {summary_path}")

Modelo resnet18 en: cuda:0

=== Entrenando resnet18 en cuda:0 (AMP) ===


Epoch 01/20 | train_loss=0.0302 | train_acc=0.9912 | val_loss=0.0001 | val_acc=1.0000 | val_f1=1.0000


Epoch 02/20 | train_loss=0.0142 | train_acc=0.9968 | val_loss=0.0019 | val_acc=0.9993 | val_f1=0.9993


Epoch 03/20 | train_loss=0.0010 | train_acc=0.9998 | val_loss=0.0000 | val_acc=1.0000 | val_f1=1.0000


Epoch 04/20 | train_loss=0.0123 | train_acc=0.9974 | val_loss=0.0024 | val_acc=1.0000 | val_f1=1.0000


...

Epoch 04/20 | train_loss=0.0018 | train_acc=0.9995 | val_loss=0.0000 | val_acc=1.0000 | val_f1=1.0000


Epoch 05/20 | train_loss=0.0052 | train_acc=0.9986 | val_loss=0.0050 | val_acc=0.9993 | val_f1=0.9993


Epoch 06/20 | train_loss=0.0021 | train_acc=0.9993 | val_loss=0.0023 | val_acc=0.9993 | val_f1=0.9993


Epoch 07/20 | train_loss=0.0005 | train_acc=0.9998 | val_loss=0.0000 | val_acc=1.0000 | val_f1=1.0000
Early stopping en epoch 7 (paciencia=5)


Mejor mobilenet_v3_small: acc=1.0000 f1_macro=1.0000 (6.2 min) | device=cuda:0
Historial guardado en: G:\Maestria\Modulo 9\Proyecto Final\docs\figures\history_mobilenet_v3_small.json
             model   params  epochs_run  minutes  accuracy  f1_macro  f1_weighted  val_loss                                                                         checkpoint
          resnet18 11179077           6     4.95       1.0       1.0          1.0  0.000101           G:\Maestria\Modulo 9\Proyecto Final\outputs\checkpoints\resnet18_best.pt
   efficientnet_b0  4013953           6     7.72       1.0       1.0          1.0  0.000038    G:\Maestria\Modulo 9\Proyecto Final\outputs\checkpoints\efficientnet_b0_best.pt
mobilenet_v3_small  1522981           7     6.19       1.0       1.0          1.0  0.000894 G:\Maestria\Modulo 9\Proyecto Final\outputs\checkpoints\mobilenet_v3_small_best.pt

Resumen guardado en: G:\Maestria\Modulo 9\Proyecto Final\outputs\comparison_summary.csv


## 6. Curvas de pérdida y precisión

Gráficos separados de **loss** y **accuracy** (train vs val) por modelo.


In [8]:
# Curvas de pérdida (train vs val)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, r in zip(axes, results):
    h = r["history"]
    ax.plot(h["epoch"], h["train_loss"], label="train_loss", marker="o", markersize=3)
    ax.plot(h["epoch"], h["val_loss"], label="val_loss", marker="s", markersize=3)
    ax.set_title(r["model"])
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle("Pérdida de entrenamiento y validación")
plt.tight_layout()
loss_path = FIGURES_DIR / "loss_curves.png"
plt.savefig(loss_path, dpi=150, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "loss_curves.png", dpi=150, bbox_inches="tight")
show_figure()
print(f"Pérdida guardada en: {loss_path}")

# Curvas de precisión (train vs val)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, r in zip(axes, results):
    h = r["history"]
    ax.plot(h["epoch"], h["train_acc"], label="train_acc", marker="o", markersize=3)
    ax.plot(h["epoch"], h["val_acc"], label="val_acc", marker="s", markersize=3)
    ax.set_title(r["model"])
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0.9, 1.01)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle("Precisión de entrenamiento y validación")
plt.tight_layout()
acc_path = FIGURES_DIR / "accuracy_curves.png"
plt.savefig(acc_path, dpi=150, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "accuracy_curves.png", dpi=150, bbox_inches="tight")
show_figure()
print(f"Precisión guardada en: {acc_path}")


Pérdida guardada en: G:\Maestria\Modulo 9\Proyecto Final\docs\figures\loss_curves.png


Precisión guardada en: G:\Maestria\Modulo 9\Proyecto Final\docs\figures\accuracy_curves.png


## 7. Matrices de confusión y reporte por clase

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, r in zip(axes, results):
    cm = confusion_matrix(r["y_true"], r["y_pred"])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=ax,
        cbar=False,
    )
    ax.set_title(f"{r['model']}\nacc={r['accuracy']:.3f} | f1={r['f1_macro']:.3f}")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
plt.tight_layout()
cm_path = FIGURES_DIR / "confusion_matrices.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
show_figure()
print(f"Matrices guardadas en: {cm_path}")

for r in results:
    print("\n" + "=" * 60)
    print(r["model"])
    print(
        classification_report(
            r["y_true"], r["y_pred"], target_names=CLASS_NAMES, digits=4, zero_division=0
        )
    )


Matrices guardadas en: G:\Maestria\Modulo 9\Proyecto Final\docs\figures\confusion_matrices.png

resnet18
              precision    recall  f1-score   support

       crack     1.0000    1.0000    1.0000       600
        hole     1.0000    1.0000    1.0000       600
      normal     1.0000    1.0000    1.0000       600
        rust     1.0000    1.0000    1.0000       600
     scratch     1.0000    1.0000    1.0000       600

    accuracy                         1.0000      3000
   macro avg     1.0000    1.0000    1.0000      3000
weighted avg     1.0000    1.0000    1.0000      3000


efficientnet_b0
              precision    recall  f1-score   support

       crack     1.0000    1.0000    1.0000       600
        hole     1.0000    1.0000    1.0000       600
      normal     1.0000    1.0000    1.0000       600
        rust     1.0000    1.0000    1.0000       600
     scratch     1.0000    1.0000    1.0000       600

    accuracy                         1.0000      3000
   macro 

## 8. Inferencia rápida con el mejor checkpoint

In [10]:
best_row = summary.iloc[0]
best_name = best_row["model"]
ckpt = torch.load(best_row["checkpoint"], map_location=DEVICE, weights_only=True)

best_model = build_model(best_name).to(DEVICE)
best_model.load_state_dict(ckpt["state_dict"])
best_model.eval()

print(f"Mejor modelo: {best_name}")
print(f"Métricas guardadas: {ckpt['metrics']}")

imgs, labs = next(iter(val_loader))
with torch.no_grad():
    logits = best_model(imgs.to(DEVICE))
    preds = logits.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, img, true_y, pred_y in zip(axes.ravel(), imgs[:8], labs[:8], preds[:8]):
    x = img.permute(1, 2, 0).numpy()
    x = np.clip(x * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)
    ok = int(true_y) == int(pred_y)
    ax.imshow(x)
    ax.set_title(
        f"T:{CLASS_NAMES[int(true_y)]}\nP:{CLASS_NAMES[int(pred_y)]}",
        color="green" if ok else "red",
        fontsize=9,
    )
    ax.axis("off")
plt.suptitle(f"Inferencia — {best_name}")
plt.tight_layout()
inf_path = FIGURES_DIR / "inference_predictions.png"
plt.savefig(inf_path, dpi=150, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "inference_predictions.png", dpi=150, bbox_inches="tight")
show_figure()
print(f"Inferencia guardada en: {inf_path}")


Mejor modelo: resnet18
Métricas guardadas: {'accuracy': 1.0, 'f1_macro': 1.0, 'f1_weighted': 1.0, 'val_loss': 0.00010139158110541757}


Inferencia guardada en: G:\Maestria\Modulo 9\Proyecto Final\docs\figures\inference_predictions.png


## 9. Exportar metadatos del experimento

In [11]:
experiment = {
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "device": str(DEVICE),
    "classes": CLASS_NAMES,
    "models": summary.to_dict(orient="records"),
}

exp_path = OUTPUT_DIR / "experiment_config.json"
with open(exp_path, "w", encoding="utf-8") as f:
    json.dump(experiment, f, indent=2, ensure_ascii=False)

print(f"Experimento guardado en: {exp_path}")
print(f"Checkpoints en: {CHECKPOINT_DIR}")
summary

Experimento guardado en: G:\Maestria\Modulo 9\Proyecto Final\outputs\experiment_config.json
Checkpoints en: G:\Maestria\Modulo 9\Proyecto Final\outputs\checkpoints


,model,params,epochs_run,minutes,accuracy,f1_macro,f1_weighted,val_loss,checkpoint
0,resnet18,11179077,6,4.95,1.0,1.0,1.0,0.000101,G:\Maestria\Modulo 9\Proyecto Final\outputs\ch...
1,efficientnet_b0,4013953,6,7.72,1.0,1.0,1.0,0.000038,G:\Maestria\Modulo 9\Proyecto Final\outputs\ch...
2,mobilenet_v3_small,1522981,7,6.19,1.0,1.0,1.0,0.000894,G:\Maestria\Modulo 9\Proyecto Final\outputs\ch...
